#### -----------------------------------------------------------------------------<br>Copyright (c) 2024, Lucid Vision Labs, Inc.
##### THE  SOFTWARE  IS  PROVIDED "AS IS",  WITHOUT  WARRANTY  OF  ANY  KIND,<br>EXPRESS  OR  IMPLIED,  INCLUDING  BUT  NOT  LIMITED  TO  THE  WARRANTIES<br>OF  MERCHANTABILITY,  FITNESS  FOR  A  PARTICULAR  PURPOSE  AND<br>NONINFRINGEMENT.  IN  NO  EVENT  SHALL  THE  AUTHORS  OR  COPYRIGHT  HOLDERS<br>BE  LIABLE  FOR  ANY  CLAIM,  DAMAGES  OR  OTHER  LIABILITY,  WHETHER  IN  AN<br>ACTION  OF  CONTRACT,  TORT  OR  OTHERWISE,  ARISING  FROM,  OUT  OF  OR  IN<br>CONNECTION  WITH  THE  SOFTWARE  OR  THE  USE  OR  OTHER  DEALINGS  IN  THE  SOFTWARE.<br>-----------------------------------------------------------------------------

In [13]:
import os
import time

from arena_api.__future__.save import Writer
from arena_api.enums import PixelFormat
from arena_api.system import system
from arena_api.buffer import BufferFactory
from arena_api.__future__.save import _xWriter, _xReader

RAW_FILE_PATH = "images\\py_acquisition_compressed_image_loading\\CompressedImage"
PNG_FILE_PATH = "images\\py_acquisition_compressed_image_loading\\DecompressedImage"

TAB1 = "  "
TAB2 = "    "

#### Acquisition: Compressed Image Loading
>	This example demonstrates how to handle compressed image data, specifically
	loading and processing from raw data files using the Arena SDK. The example
	includes steps to configure the camera, acquire a compressed image, save
	the raw file, load the raw file, decompress the data, and save the decompressed
	image.

In [14]:
tries = 0
tries_max = 6
sleep_time_secs = 10
while tries < tries_max:  # Wait for device for 60 seconds
	devices = system.create_device()
	if not devices:
		print(
			f'Try {tries+1} of {tries_max}: waiting for {sleep_time_secs} '
			f'secs for a device to be connected!')
		for sec_count in range(sleep_time_secs):
			time.sleep(1)
			print(f'{sec_count + 1 } seconds passed ',
				'.' * sec_count, end='\r')
		tries += 1
	else:
		print(f'Created {len(devices)} device(s)')
		device = system.select_device(devices)
		break
else:
	raise Exception(f'No device found! Please connect a device and run '
					f'the example again.')

print(f'Device used in the example:\n\t{device}')

Created 1 device(s)
  Only one device detected:  ('1c:0f:af:3f:55:a4', 'PHX064S-M', '', '169.254.165.85')
    Automatically selecting this device.
Device used in the example:
	('1c:0f:af:3f:55:a4', 'PHX064S-M', '', '169.254.165.85')


##### Enable stream auto negotiate packet size
>   Setting the stream packet size is done before starting the stream.
	Setting the stream to automatically negotiate packet size instructs the
	camera to receive the largest packet size that the system will allow.
	This generally increases frame rate and results in fewer interrupts per
	image, thereby reducing CPU load on the host system. Ethernet settings
	may also be manually changed to allow for a larger packet size.

In [15]:
tl_stream_nodemap = device.tl_stream_nodemap

tl_stream_nodemap['StreamAutoNegotiatePacketSize'].value = True

##### Enable stream packet resend
>   Enable stream packet resend before starting the stream. Images are sent
	from the camera to the host in packets using UDP protocol, which
	includes a header image number, packet number, and timestamp
	information. If a packet is missed while receiving an image, a packet
	resend is requested and this information is used to retrieve and
	redeliver the missing packet in the correct order.

In [16]:
tl_stream_nodemap['StreamPacketResendEnable'].value = True

#### Set features before streaming
>   Set PixelFormat to QOI_Mono8

In [17]:
# Get device nodemap
nodemap = device.nodemap

# Iterate through the PixelFormat enum and check for "QOI_Mono8"
node = nodemap.get_node('PixelFormat')
entries = node.enumentry_names
found = False
for e in entries:
	if (e == 'QOI_Mono8'):
		found = True
		break

if (not found):
	print(f'QOI_Mono8 is not available in the PixelFormat enumeration for this camera.\n')

# Get initial node values in order to return their values at the end of the example
pixel_format_initial = node.value
print(f"Initial PixelFormat value: {pixel_format_initial}")

Initial PixelFormat value: QOI_Mono8


In [18]:
# - PixelFormat to QOI_Mono8
pixel_format_setting = 'QOI_Mono8'
print(f'Setting pixel format to { pixel_format_setting }\n')
node.value = pixel_format_setting

Setting pixel format to QOI_Mono8



#### Start stream and grab images
>   - Starting stream with one buffer, grabbing one image.
>   - Printing the uncompressed size and compressed size for comparison.
>   - Saving compresed image (in PixelFormat QOI_Mono8) in its raw state. 
>	- Loading saved image and then decompressing image to Mono8.
>   - Must requeue buffer in order to prevent memory leaks.

In [19]:
def save_compressed_image(buffer, index):

	print(f'{TAB1}Save compressed input image data to', end=' ')

	filename = RAW_FILE_PATH + str(index) + ".raw"

	# Save function for .raw file
	_xWriter.SaveRawData(filename, buffer.compressed_image_pdata, buffer.size_filled)
	print(f'{filename}')

In [20]:
with device.start_stream(1):

	print(f'Stream started with 1 buffer')

	for i in range(0, 10):
		# Get compressed image
		print(f'Get compressed image {i}')
		buffer = device.get_buffer()

		# Get compressed image size
		compressed_image_size = buffer.size_filled
		print(f'{TAB1}Compressed image {i} size: {compressed_image_size} bytes')

		# Save the raw image
		save_compressed_image(buffer, i)

		# Re-queue the image buffer
		device.requeue_buffer(buffer)

Stream started with 1 buffer
Get compressed image 0
  Compressed image 0 size: 2405640 bytes
  Save compressed input image data to images\py_acquisition_compressed_image_loading\CompressedImage0.raw
Get compressed image 1
  Compressed image 1 size: 2408840 bytes
  Save compressed input image data to images\py_acquisition_compressed_image_loading\CompressedImage1.raw
Get compressed image 2
  Compressed image 2 size: 2410492 bytes
  Save compressed input image data to images\py_acquisition_compressed_image_loading\CompressedImage2.raw
Get compressed image 3
  Compressed image 3 size: 2409100 bytes
  Save compressed input image data to images\py_acquisition_compressed_image_loading\CompressedImage3.raw
Get compressed image 4
  Compressed image 4 size: 2407000 bytes
  Save compressed input image data to images\py_acquisition_compressed_image_loading\CompressedImage4.raw
Get compressed image 5
  Compressed image 5 size: 2408012 bytes
  Save compressed input image data to images\py_acquisiti

>	Stops stream and prevents memory leaks

In [21]:
# Stop stream
print(f'Stopping stream')
device.stop_stream()

Stopping stream


>	Loads image and decompresses it, saving it as a file

In [22]:
print(f'Load and process image')

begin = time.time()

for i in range(0, 10):
	
	in_filename = RAW_FILE_PATH + str(i) + ".raw"

	# Read raw file size
	size = os.path.getsize(in_filename);

	# Read file
	pdata = _xReader.LoadRawData(in_filename, size)

	# Load file into compressed image
	compressed_image = BufferFactory.create_compressed_image(pdata, size, PixelFormat.QOI_Mono8)

	# Decompress image
	decompressed_image = BufferFactory.decompress_image(compressed_image)

	# Get and print size for comparison
	decompressed_image_size = decompressed_image.size_filled
	print(f'{TAB1}Decompressed image {i} size: {decompressed_image_size} bytes')

	out_filename = PNG_FILE_PATH + str(i) + ".png"

	# Save the decompressed image
	writer = Writer.from_buffer(decompressed_image)
	writer.pattern = out_filename
	writer.save(decompressed_image)
	print(f'{TAB1}Image saved {writer.saved_images[-1]}')

	# Destroy buffer to avoid memory leaks
	BufferFactory.destroy(decompressed_image)
	BufferFactory.destroy_compressed_image(compressed_image)

end = time.time()
print(f'Time to decompress 10 images (sec) = {end - begin}')

Load and process image
  Decompressed image 0 size: 6291456 bytes
  Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_loading\DecompressedImage0.png
  Decompressed image 1 size: 6291456 bytes
  Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_loading\DecompressedImage1.png
  Decompressed image 2 size: 6291456 bytes
  Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_loading\DecompressedImage2.png
  Decompressed image 3 size: 6291456 bytes
  Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_loading\DecompressedImage3.png
  Decompressed image 4 size: 6291456 bytes
  Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_loading\DecompressedImage4.png
  Decompressed im

>	Resets node values to initial values

In [23]:
node.value = pixel_format_initial

##### Clean up
> Destroy device. This call is optional and will automatically be
  called for any remaining devices when the system module is unloading.

In [24]:
system.destroy_device()
print(f'Destroyed all created devices')

Destroyed all created devices
